In [22]:
# ==============================================================================
# 🔬 DATALOOM — INSTALLATION & PROJECT SETUP
# ==============================================================================

!pip install -q \
    polars \
    duckdb \
    openai \
    instructor \
    pydantic \
    chromadb \
    langgraph \
    langchain-core \
    langchain-openai \
    pandas \
    kagglehub \
    pandera \
    sentence-transformers \
    fastapi \
    uvicorn \
    gradio


# ==============================================================================
# PROJECT DIRECTORIES
# ==============================================================================

from pathlib import Path

directories = [
    "src",
    "data/raw",
    "data/processed",
    "chroma_db"
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("✔ Dependencies installed")
print("✔ Project directories created")

✔ Dependencies installed
✔ Project directories created


In [23]:
# ==============================================================================
# ⚙️ DATALOOM — CONFIGURATION
# ==============================================================================

from pathlib import Path


config_code = r'''
import os
from pathlib import Path


# ==============================================================================
# PROJECT PATHS
# ==============================================================================

BASE_DIR = Path(__file__).resolve().parent.parent

DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
CHROMA_DB_DIR = BASE_DIR / "chroma_db"


# ==============================================================================
# OPENROUTER CONFIGURATION
# ==============================================================================

API_KEY = "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"

BASE_URL = "https://openrouter.ai/api/v1"

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"


# ==============================================================================
# LANGSMITH OBSERVABILITY
# ==============================================================================

os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_ENDPOINT"] = (
    "https://api.smith.langchain.com"
)

os.environ["LANGCHAIN_API_KEY"] = "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"

os.environ["LANGCHAIN_PROJECT"] = "DataLoom-Production"


# ==============================================================================
# VALIDATION
# ==============================================================================

def validate_config():

    if not API_KEY or API_KEY.startswith("YOUR_"):
        raise EnvironmentError(
            "OpenRouter API key is not configured."
        )

    print("✔ OpenRouter configuration validated")
    print(f"✔ Model: {MODEL_NAME}")
    print("✔ LangSmith tracing enabled")
'''


Path("src").mkdir(parents=True, exist_ok=True)

Path("src/config.py").write_text(
    config_code,
    encoding="utf-8"
)

print("✔ src/config.py created successfully")

✔ src/config.py created successfully


In [24]:
# ==============================================================================
# 🔄 RELOAD CONFIGURATION MODULE
# ==============================================================================

import sys
import importlib

if "src.config" in sys.modules:
    importlib.reload(sys.modules["src.config"])

from src import config

print("✔ src.config reloaded")

print(
    "✔ validate_config exists:",
    hasattr(config, "validate_config")
)

✔ src.config reloaded
✔ validate_config exists: True


In [25]:
# ==============================================================================
# 🔐 VALIDATE CONFIGURATION
# ==============================================================================

from src.config import validate_config

validate_config()

print("✔ Configuration is ready")

✔ OpenRouter configuration validated
✔ Model: meta-llama/llama-3.3-70b-instruct
✔ LangSmith tracing enabled
✔ Configuration is ready


In [26]:
# ==============================================================================
# 📥 DATALOOM — DATA INGESTION
# ==============================================================================

from pathlib import Path


ingestion_code = r'''
import duckdb
import polars as pl

from pathlib import Path

from src.config import PROCESSED_DATA_DIR


class ArxivIngestion:

    def __init__(self, input_file: str):
        self.input_file = Path(input_file)

    def process(self, limit: int = 5000):

        if not self.input_file.exists():
            raise FileNotFoundError(
                f"Dataset not found: {self.input_file}"
            )

        conn = duckdb.connect(":memory:")

        try:

            query = f"""
                SELECT
                    id,
                    title,
                    abstract,
                    categories,
                    update_date
                FROM read_json_auto(
                    '{self.input_file.as_posix()}'
                )
                WHERE
                    id IS NOT NULL
                    AND title IS NOT NULL
                    AND abstract IS NOT NULL
                    AND categories IS NOT NULL
                LIMIT {int(limit)}
            """

            df = conn.execute(query).pl()

        finally:
            conn.close()


        # ----------------------------------------------------------------------
        # CLEANING
        # ----------------------------------------------------------------------

        df = (
            df
            .with_columns([
                pl.col("title")
                .cast(pl.Utf8)
                .str.replace_all(r"\s+", " ")
                .str.strip_chars(),

                pl.col("abstract")
                .cast(pl.Utf8)
                .str.replace_all(r"\s+", " ")
                .str.strip_chars(),

                pl.col("categories")
                .cast(pl.Utf8)
                .str.replace_all(r"\s+", " ")
                .str.strip_chars(),
            ])
            .unique(subset=["id"])
        )


        # ----------------------------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------------------------

        output_file = (
            PROCESSED_DATA_DIR /
            "silver_arxiv.parquet"
        )

        df.write_parquet(output_file)

        print(f"✔ Records loaded: {len(df)}")
        print(f"✔ Output: {output_file}")

        return df
'''

Path("src/ingestion.py").write_text(
    ingestion_code,
    encoding="utf-8"
)

print("✔ src/ingestion.py created")

✔ src/ingestion.py created


In [27]:
# ==============================================================================
# 📊 DATALOOM — ANALYTICS ENGINE
# ==============================================================================

from pathlib import Path


analytics_code = r'''
import duckdb

from src.config import PROCESSED_DATA_DIR


class AnalyticsEngine:

    def __init__(self):

        self.db = duckdb.connect(":memory:")

        self.parquet_file = (
            PROCESSED_DATA_DIR /
            "silver_arxiv.parquet"
        )

        self.db.execute(
            f"""
            CREATE OR REPLACE VIEW silver_papers AS
            SELECT *
            FROM read_parquet(
                '{self.parquet_file.as_posix()}'
            )
            """
        )


    # ==========================================================================
    # DATASET OVERVIEW
    # ==========================================================================

    def get_dataset_overview(self):

        result = self.db.execute(
            """
            SELECT
                COUNT(*) AS total_papers,
                COUNT(DISTINCT id) AS unique_papers
            FROM silver_papers
            """
        ).fetchone()

        return {
            "total_papers": result[0],
            "unique_papers": result[1]
        }


    # ==========================================================================
    # CATEGORY ANALYSIS
    # ==========================================================================

    def get_paper_counts_by_category(
        self,
        limit: int = 10
    ):

        return self.db.execute(
            f"""
            SELECT
                categories,
                COUNT(*) AS paper_count
            FROM silver_papers
            GROUP BY categories
            ORDER BY paper_count DESC
            LIMIT {int(limit)}
            """
        ).df()


    # ==========================================================================
    # TEMPORAL TRENDS
    # ==========================================================================

    def get_temporal_trends(self):

        return self.db.execute(
            """
            SELECT
                EXTRACT(YEAR FROM CAST(update_date AS DATE))
                    AS year,
                COUNT(*) AS paper_count
            FROM silver_papers
            WHERE update_date IS NOT NULL
            GROUP BY year
            ORDER BY year
            """
        ).df()


    # ==========================================================================
    # CATEGORY SEARCH
    # ==========================================================================

    def search_categories(
        self,
        keyword: str,
        limit: int = 10
    ):

        keyword = keyword.replace("'", "''")

        return self.db.execute(
            f"""
            SELECT
                categories,
                COUNT(*) AS paper_count
            FROM silver_papers
            WHERE LOWER(categories)
                LIKE '%{keyword.lower()}%'
            GROUP BY categories
            ORDER BY paper_count DESC
            LIMIT {int(limit)}
            """
        ).df()


    # ==========================================================================
    # CLOSE
    # ==========================================================================

    def close(self):

        self.db.close()
'''

Path("src/analytics.py").write_text(
    analytics_code,
    encoding="utf-8"
)

print("✔ src/analytics.py created")

✔ src/analytics.py created


In [28]:
# ==============================================================================
# 🔎 DATALOOM — VECTOR STORE & RERANKING
# ==============================================================================

from pathlib import Path


vector_store_code = r'''
import chromadb

from sentence_transformers import CrossEncoder

from src.config import CHROMA_DB_DIR


class VectorStoreManager:

    def __init__(self):

        # ======================================================================
        # CHROMADB
        # ======================================================================

        self.client = chromadb.PersistentClient(
            path=str(CHROMA_DB_DIR)
        )

        self.collection = (
            self.client.get_or_create_collection(
                name="arxiv_papers"
            )
        )

        # ======================================================================
        # LAZY RERANKER
        # ======================================================================
        # CrossEncoder is NOT loaded during indexing.
        # It will only be loaded when search() is called.

        self.reranker = None

        print(
            "✔ ChromaDB initialized"
        )

        print(
            f"✔ Existing documents: "
            f"{self.collection.count()}"
        )


    # ==========================================================================
    # LOAD RERANKER ONLY WHEN NEEDED
    # ==========================================================================

    def _load_reranker(self):

        if self.reranker is None:

            print(
                "\n🔄 Loading CrossEncoder re-ranker..."
            )

            self.reranker = CrossEncoder(
                "cross-encoder/ms-marco-MiniLM-L-6-v2"
            )

            print(
                "✔ CrossEncoder loaded"
            )


    # ==========================================================================
    # INDEX PAPERS
    # ==========================================================================

    def add_papers(
        self,
        df,
        batch_size: int = 500
    ):

        total = len(df)

        print(
            f"\n📥 Preparing {total:,} papers for indexing..."
        )

        for start in range(
            0,
            total,
            batch_size
        ):

            batch = df.slice(
                start,
                batch_size
            )

            ids = (
                batch["id"]
                .cast(str)
                .to_list()
            )

            documents = (
                batch["title"].cast(str)
                + "\n\n"
                + batch["abstract"].cast(str)
            ).to_list()

            metadatas = []

            for row in batch.iter_rows(
                named=True
            ):

                metadatas.append({

                    "title": str(
                        row["title"]
                    ),

                    "categories": str(
                        row["categories"]
                    ),

                    "update_date": str(
                        row["update_date"]
                    )
                })


            # ------------------------------------------------------------------
            # UPSERT
            # ------------------------------------------------------------------

            self.collection.upsert(

                ids=ids,

                documents=documents,

                metadatas=metadatas
            )


            processed = min(
                start + batch_size,
                total
            )

            print(
                f"   ✔ Indexed "
                f"{processed:,}/{total:,}"
            )


        print(
            f"\n✔ Indexing completed"
        )

        print(
            f"✔ Collection size: "
            f"{self.collection.count():,}"
        )


    # ==========================================================================
    # SEARCH
    # ==========================================================================

    def search(
        self,
        query: str,
        n_results: int = 5
    ):

        # ----------------------------------------------------------------------
        # Load CrossEncoder ONLY NOW
        # ----------------------------------------------------------------------

        self._load_reranker()


        # ----------------------------------------------------------------------
        # Candidate retrieval
        # ----------------------------------------------------------------------

        candidate_count = max(
            n_results * 2,
            10
        )

        results = self.collection.query(

            query_texts=[query],

            n_results=candidate_count
        )


        if not results["documents"]:

            return []


        documents = results["documents"][0]

        ids = results["ids"][0]

        metadatas = results["metadatas"][0]


        if not documents:

            return []


        # ----------------------------------------------------------------------
        # CROSS ENCODER RERANKING
        # ----------------------------------------------------------------------

        pairs = [

            [query, document]

            for document in documents

        ]


        scores = self.reranker.predict(
            pairs
        )


        ranked = sorted(

            zip(
                ids,
                documents,
                metadatas,
                scores
            ),

            key=lambda x: float(x[3]),

            reverse=True
        )


        # ----------------------------------------------------------------------
        # FINAL RESULTS
        # ----------------------------------------------------------------------

        output = []


        for (
            paper_id,
            document,
            metadata,
            score
        ) in ranked[:n_results]:

            output.append({

                "id": paper_id,

                "document": document,

                "metadata": metadata,

                "relevance_score": float(
                    score
                )

            })


        return output


    # ==========================================================================
    # COLLECTION INFO
    # ==========================================================================

    def count(self):

        return self.collection.count()
'''


Path("src").mkdir(
    parents=True,
    exist_ok=True
)


Path(
    "src/vector_store.py"
).write_text(
    vector_store_code,
    encoding="utf-8"
)


print(
    "✔ src/vector_store.py updated"
)

print(
    "✔ CrossEncoder will now load ONLY during search"
)

✔ src/vector_store.py updated
✔ CrossEncoder will now load ONLY during search


In [29]:
# ==============================================================================
# 🤖 DATALOOM — LANGGRAPH AGENT
# ==============================================================================

from pathlib import Path


agent_code = r'''
from typing import TypedDict, Literal

from openai import OpenAI
import instructor

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from pydantic import BaseModel, Field

from src.config import (
    API_KEY,
    BASE_URL,
    MODEL_NAME
)

from src.analytics import AnalyticsEngine
from src.vector_store import VectorStoreManager


# ==============================================================================
# CLIENTS
# ==============================================================================

# Standard OpenAI-compatible client
llm_client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)


# Instructor client
# Used ONLY for structured intent classification
instructor_client = instructor.from_openai(
    llm_client
)


# ==============================================================================
# STRUCTURED INTENT
# ==============================================================================

class IntentClassification(BaseModel):

    intent: Literal[
        "sql",
        "rag",
        "hybrid"
    ] = Field(
        description=(
            "sql = statistical or dataset analytics, "
            "rag = semantic paper search, "
            "hybrid = both analytics and paper retrieval"
        )
    )


# ==============================================================================
# AGENT STATE
# ==============================================================================

class AgentState(TypedDict, total=False):

    query: str

    intent: str

    sql_results: dict

    rag_results: list

    final_response: str


# ==============================================================================
# COMPONENTS
# ==============================================================================

analytics = AnalyticsEngine()

vector_store = VectorStoreManager()


# ==============================================================================
# ROUTER
# ==============================================================================

def intent_router(
    state: AgentState
):

    query = state["query"]

    classification = (
        instructor_client.chat.completions.create(
            model=MODEL_NAME,

            response_model=IntentClassification,

            messages=[
                {
                    "role": "system",
                    "content": """
You are the routing controller for DataLoom.

Classify the user query into exactly one category:

sql:
Questions requiring dataset statistics,
counts, categories, trends, or numerical analysis.

rag:
Questions requiring semantic retrieval
of research papers.

hybrid:
Questions requiring both dataset analytics
and semantic paper retrieval.

Return only the structured classification.
"""
                },

                {
                    "role": "user",
                    "content": query
                }
            ]
        )
    )

    intent = classification.intent

    print(
        f"[Router] → {intent.upper()}"
    )

    return {
        "intent": intent
    }


# ==============================================================================
# SQL EXECUTOR
# ==============================================================================

def sql_executor(
    state: AgentState
):

    overview = (
        analytics
        .get_dataset_overview()
    )

    categories = (
        analytics
        .get_paper_counts_by_category(
            limit=10
        )
    )

    trends = (
        analytics
        .get_temporal_trends()
    )

    results = {

        "overview": overview,

        "top_categories":
            categories.to_dict(
                orient="records"
            ),

        "temporal_trends":
            trends.to_dict(
                orient="records"
            )
    }

    print(
        "[SQL] Analytics executed successfully."
    )

    return {
        "sql_results": results
    }


# ==============================================================================
# RAG EXECUTOR
# ==============================================================================

def rag_executor(
    state: AgentState
):

    results = vector_store.search(
        state["query"],
        n_results=5
    )

    print(
        f"[RAG] Retrieved {len(results)} papers."
    )

    return {
        "rag_results": results
    }


# ==============================================================================
# SYNTHESIZER
# ==============================================================================

def synthesizer(
    state: AgentState
):

    sql_data = state.get(
        "sql_results",
        {}
    )

    rag_data = state.get(
        "rag_results",
        []
    )


    # --------------------------------------------------------------------------
    # BUILD RETRIEVED PAPER CONTEXT
    # --------------------------------------------------------------------------

    source_text = ""

    for i, paper in enumerate(
        rag_data,
        start=1
    ):

        metadata = paper.get(
            "metadata",
            {}
        )

        source_text += f"""
Paper {i}:
Title: {metadata.get("title", "Unknown")}
Categories: {metadata.get("categories", "Unknown")}
Relevance Score: {paper.get("relevance_score", 0)}

Content:
{paper.get("document", "")}

---
"""


    # --------------------------------------------------------------------------
    # SYNTHESIS PROMPT
    # --------------------------------------------------------------------------

    prompt = f"""
You are DataLoom, an AI Research Intelligence assistant.

Answer the user's question using ONLY the
provided dataset analytics and retrieved papers.

IMPORTANT RULES:

1. Never invent statistics.
2. Never invent papers.
3. Never claim these results represent
   the entire arXiv dataset.
4. The current demo uses a bounded subset
   of 5,000 indexed research records.
5. Clearly distinguish dataset statistics
   from semantic paper retrieval.
6. Mention paper titles when papers are used.
7. If the available data is insufficient,
   explicitly say so.
8. Be concise but informative.

USER QUESTION:
{state["query"]}

DATASET ANALYTICS:
{sql_data}

RETRIEVED PAPERS:
{source_text}
"""


    # --------------------------------------------------------------------------
    # NORMAL LLM CALL
    # --------------------------------------------------------------------------

    response = llm_client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise research "
                    "intelligence assistant."
                )
            },

            {
                "role": "user",
                "content": prompt
            }
        ]
    )


    # --------------------------------------------------------------------------
    # EXTRACT RESPONSE
    # --------------------------------------------------------------------------

    answer_text = (
        response.choices[0]
        .message
        .content
    )


    route = state["intent"].upper()

    answer = (
        f"**🔀 DataLoom Route:** {route}\n\n"
        f"{answer_text}"
    )


    return {
        "final_response": answer
    }


# ==============================================================================
# ROUTING LOGIC
# ==============================================================================

def route_from_intent(
    state: AgentState
):

    return state["intent"]


def route_after_sql(
    state: AgentState
):

    if state["intent"] == "hybrid":

        return "rag_executor"

    return "synthesizer"


# ==============================================================================
# GRAPH
# ==============================================================================

workflow = StateGraph(
    AgentState
)


workflow.add_node(
    "router",
    intent_router
)

workflow.add_node(
    "sql_executor",
    sql_executor
)

workflow.add_node(
    "rag_executor",
    rag_executor
)

workflow.add_node(
    "synthesizer",
    synthesizer
)


workflow.set_entry_point(
    "router"
)


# ------------------------------------------------------------------------------
# ROUTER → NEXT STEP
# ------------------------------------------------------------------------------

workflow.add_conditional_edges(

    "router",

    route_from_intent,

    {
        "sql": "sql_executor",

        "rag": "rag_executor",

        "hybrid": "sql_executor"
    }
)


# ------------------------------------------------------------------------------
# SQL → SYNTHESIS OR RAG
# ------------------------------------------------------------------------------

workflow.add_conditional_edges(

    "sql_executor",

    route_after_sql,

    {
        "rag_executor": "rag_executor",

        "synthesizer": "synthesizer"
    }
)


# ------------------------------------------------------------------------------
# RAG → SYNTHESIS
# ------------------------------------------------------------------------------

workflow.add_edge(
    "rag_executor",
    "synthesizer"
)


# ------------------------------------------------------------------------------
# SYNTHESIS → END
# ------------------------------------------------------------------------------

workflow.add_edge(
    "synthesizer",
    END
)


# ==============================================================================
# MEMORY
# ==============================================================================

memory = MemorySaver()

app = workflow.compile(
    checkpointer=memory
)


# ==============================================================================
# PUBLIC RUN FUNCTION
# ==============================================================================

def run(
    query: str,
    session_id: str = "default"
):

    result = app.invoke(

        {
            "query": query
        },

        config={
            "configurable": {
                "thread_id": session_id
            }
        }
    )

    return result["final_response"]
'''


Path("src").mkdir(
    parents=True,
    exist_ok=True
)

Path(
    "src/agent.py"
).write_text(
    agent_code,
    encoding="utf-8"
)

print("✔ src/agent.py updated")
print("✔ Instructor used only for structured intent")
print("✔ Standard OpenAI client used for synthesis")
print("✔ SQL / RAG / Hybrid routing preserved")

✔ src/agent.py updated
✔ Instructor used only for structured intent
✔ Standard OpenAI client used for synthesis
✔ SQL / RAG / Hybrid routing preserved


In [30]:
# ==============================================================================
# 🚀 DATALOOM — FASTAPI
# ==============================================================================

from pathlib import Path


main_code = r'''
from fastapi import FastAPI
from pydantic import BaseModel

from src.agent import run


app = FastAPI(
    title="DataLoom AI Research Intelligence API",
    version="1.0.0"
)


class QueryRequest(BaseModel):

    query: str

    session_id: str = "default"


@app.get("/")
def root():

    return {
        "project": "DataLoom",
        "status": "running"
    }


@app.post("/chat")
def chat(
    request: QueryRequest
):

    response = run(
        query=request.query,
        session_id=request.session_id
    )

    return {
        "query": request.query,
        "response": response
    }
'''

Path("src/main.py").write_text(
    main_code,
    encoding="utf-8"
)

print("✔ src/main.py created")

✔ src/main.py created


In [33]:
# ==============================================================================
# 📥 DOWNLOAD & PROCESS ARXIV DATASET
# ==============================================================================

import kagglehub
from pathlib import Path

from src.ingestion import ArxivIngestion


# ==============================================================================
# DOWNLOAD
# ==============================================================================

dataset_path = kagglehub.dataset_download(
    "Cornell-University/arxiv"
)

dataset_path = Path(dataset_path)

print(
    f"✔ Dataset downloaded to:\n{dataset_path}"
)


# ==============================================================================
# FIND JSON FILE
# ==============================================================================

json_files = list(
    dataset_path.rglob(
        "arxiv-metadata-oai-snapshot.json"
    )
)

if not json_files:

    raise FileNotFoundError(
        "arxiv-metadata-oai-snapshot.json "
        "was not found."
    )


json_file = json_files[0]

print(
    f"✔ Metadata file found:\n{json_file}"
)


# ==============================================================================
# PROCESS
# ==============================================================================

ingestion = ArxivIngestion(
    str(json_file)
)

df = ingestion.process(
    limit=5000
)

print(
    f"\n✔ Final processed dataset: {df.shape}"
)

✔ Dataset downloaded to:
/kaggle/input/datasets/Cornell-University/arxiv
✔ Metadata file found:
/kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json
✔ Records loaded: 5000
✔ Output: /kaggle/working/data/processed/silver_arxiv.parquet

✔ Final processed dataset: (5000, 5)


In [35]:
# ==============================================================================
# 📁 CHECK DATALOOM DATA FILES
# ==============================================================================

from pathlib import Path

processed_dir = Path("data/processed")
parquet_file = processed_dir / "silver_arxiv.parquet"

print("=" * 80)
print("📁 DATALOOM — DATA FILE CHECK")
print("=" * 80)

print(f"\nProcessed directory:")
print(processed_dir.resolve())

print(f"\nParquet file:")
print(parquet_file.resolve())

if parquet_file.exists():
    print("\n✔ silver_arxiv.parquet exists")
    print(f"✔ Size: {parquet_file.stat().st_size / (1024 * 1024):.2f} MB")
else:
    print("\n❌ silver_arxiv.parquet NOT FOUND")

📁 DATALOOM — DATA FILE CHECK

Processed directory:
/kaggle/working/data/processed

Parquet file:
/kaggle/working/data/processed/silver_arxiv.parquet

✔ silver_arxiv.parquet exists
✔ Size: 1.51 MB


In [34]:
# ==============================================================================
# 🔄 RELOAD DATALOOM MODULES
# ==============================================================================

import importlib

import src.config
import src.ingestion
import src.analytics
import src.vector_store
import src.agent

importlib.reload(src.config)
importlib.reload(src.ingestion)
importlib.reload(src.analytics)
importlib.reload(src.vector_store)
importlib.reload(src.agent)

print("✔ All DataLoom modules reloaded")

✔ ChromaDB initialized
✔ Existing documents: 0
✔ ChromaDB initialized
✔ Existing documents: 0
✔ All DataLoom modules reloaded


In [36]:
# ==============================================================================
# 🔄 RELOAD AGENT
# ==============================================================================

import importlib

import src.agent

importlib.reload(
    src.agent
)

print("✔ src.agent reloaded")

✔ ChromaDB initialized
✔ Existing documents: 0
✔ src.agent reloaded


In [37]:
# ==============================================================================
# 🧪 DATALOOM — SYSTEM VALIDATION
# ==============================================================================

from src.agent import run


tests = [

    (
        "SQL",
        "What are the top 5 research categories?"
    ),

    (
        "RAG",
        "Find papers discussing neural networks."
    ),

    (
        "HYBRID",
        "What are the top research categories, "
        "and can you find papers discussing "
        "neural networks or language models?"
    )
]


for test_name, query in tests:

    print("\n")
    print("=" * 90)
    print(f"🧪 TEST: {test_name}")
    print("=" * 90)

    try:

        result = run(
            query=query,
            session_id=f"test_{test_name.lower()}"
        )

        print(result)

    except Exception as e:

        print(
            f"❌ {test_name} failed:"
        )

        print(
            type(e).__name__,
            str(e)
        )



🧪 TEST: SQL
[Router] → SQL
[SQL] Analytics executed successfully.
**🔀 DataLoom Route:** SQL

Based on the provided dataset analytics, the top 5 research categories are:

1. astro-ph (915 papers)
2. hep-ph (231 papers)
3. quant-ph (205 papers)
4. hep-th (193 papers)
5. cond-mat.mtrl-sci (111 papers)

Note that these statistics are derived from the dataset analytics and represent the distribution of paper categories within the bounded subset of 5,000 indexed research records. No papers were retrieved for this query, so no paper titles are mentioned.


🧪 TEST: RAG
[Router] → RAG

🔄 Loading CrossEncoder re-ranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✔ CrossEncoder loaded


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 69.0MiB/s]


[RAG] Retrieved 0 papers.
**🔀 DataLoom Route:** RAG

Unfortunately, the dataset analytics are empty and no papers have been retrieved. Therefore, I do not have sufficient data to provide papers discussing neural networks. If you would like to provide more context or clarify the search query, I'll do my best to assist you.


🧪 TEST: HYBRID
[Router] → HYBRID
[SQL] Analytics executed successfully.
[RAG] Retrieved 0 papers.
**🔀 DataLoom Route:** HYBRID

Based on the provided dataset analytics, the top research categories are: 

1. astro-ph (915 papers)
2. hep-ph (231 papers)
3. quant-ph (205 papers)
4. hep-th (193 papers)
5. cond-mat.mtrl-sci (111 papers)

As for papers discussing neural networks or language models, I couldn't find any relevant papers in the retrieved papers list, which is currently empty. If the available data is updated with relevant papers, I can try to provide more information. For now, the available data is insufficient to answer this part of the question.


In [38]:
from IPython.display import Markdown, display

display(Markdown("""
# 🔬 DataLoom — AI Research Intelligence Platform

## Architecture

```text
                    ┌─────────────────────┐
                    │    ArXiv Dataset    │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   DuckDB + Polars   │
                    │ Data Ingestion      │
                    │ Cleaning            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │     Parquet         │
                    │ Silver Data Layer   │
                    └───────┬─────┬───────┘
                            │     │
                    ┌───────▼─┐ ┌─▼──────────┐
                    │ DuckDB  │ │ ChromaDB   │
                    │Analytics│ │Vector Store│
                    └────┬────┘ └─────┬──────┘
                         │             │
                         │       ┌─────▼─────────┐
                         │       │ CrossEncoder  │
                         │       │ Re-ranking    │
                         │       └─────┬─────────┘
                         │             │
                         └──────┬──────┘
                                ▼
                       ┌─────────────────┐
                       │    LangGraph    │
                       │     Router      │
                       └───────┬─────────┘
                               │
                    ┌──────────┼──────────┐
                    ▼          ▼          ▼
                  SQL         RAG       HYBRID
                    │          │          │
                    └──────────┼──────────┘
                               ▼
                       ┌─────────────────┐
                       │  LLM Synthesis  │
                       └────────┬────────┘
                                ▼
                       ┌─────────────────┐
                       │  Gradio Chat UI │
                       └─────────────────┘
"""))


# 🔬 DataLoom — AI Research Intelligence Platform

## Architecture

```text
                    ┌─────────────────────┐
                    │    ArXiv Dataset    │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   DuckDB + Polars   │
                    │ Data Ingestion      │
                    │ Cleaning            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │     Parquet         │
                    │ Silver Data Layer   │
                    └───────┬─────┬───────┘
                            │     │
                    ┌───────▼─┐ ┌─▼──────────┐
                    │ DuckDB  │ │ ChromaDB   │
                    │Analytics│ │Vector Store│
                    └────┬────┘ └─────┬──────┘
                         │             │
                         │       ┌─────▼─────────┐
                         │       │ CrossEncoder  │
                         │       │ Re-ranking    │
                         │       └─────┬─────────┘
                         │             │
                         └──────┬──────┘
                                ▼
                       ┌─────────────────┐
                       │    LangGraph    │
                       │     Router      │
                       └───────┬─────────┘
                               │
                    ┌──────────┼──────────┐
                    ▼          ▼          ▼
                  SQL         RAG       HYBRID
                    │          │          │
                    └──────────┼──────────┘
                               ▼
                       ┌─────────────────┐
                       │  LLM Synthesis  │
                       └────────┬────────┘
                                ▼
                       ┌─────────────────┐
                       │  Gradio Chat UI │
                       └─────────────────┘


In [ ]:
# ==============================================================================
# 🎨 DATALOOM — GRADIO DEMO
# ==============================================================================

import gradio as gr

from src.agent import run


def chat_function(
    message,
    history
):

    try:

        return run(
            query=message,
            session_id="gradio_demo"
        )

    except Exception as e:

        return (
            "❌ An error occurred:\n\n"
            f"`{type(e).__name__}: {str(e)}`"
        )


demo = gr.ChatInterface(

    fn=chat_function,

    title=(
        "🔬 DataLoom — "
        "AI Research Intelligence"
    ),

    description="""
DataLoom is an AI Research Intelligence platform
combining:

• DuckDB analytics
• Polars data processing
• ChromaDB semantic retrieval
• CrossEncoder reranking
• LangGraph agentic routing
• SQL + RAG + Hybrid workflows
• LLM-powered synthesis

📊 Demo dataset:
5,000 indexed arXiv research records.
""",

    examples=[

        "What are the top research categories?",

        "Find papers discussing neural networks.",

        "Find papers about transformer architectures.",

        "What are the main research areas in this dataset?",

        "What are the top categories, "
        "and find papers about language models.",

        "Show me research trends and "
        "papers related to deep learning."

    ],

    type="messages",

    chatbot=gr.Chatbot(
        type="messages",
        height=600,
        show_copy_button=True
    )
)


demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_58/2670976518.py:75: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot=gr.Chatbot(
/tmp/ipykernel_58/2670976518.py:75: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://19766665148b454463.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[Router] → SQL
[SQL] Analytics executed successfully.
